# Rag from Scracth

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# LANGCHAIN_TRACING_V2: enables tracing of runs in LangSmith
# LANGSMITH_ENDPOINT: URL of the LangSmith service traces are sent to (depends on the ubication)
# LANGSMITH_API_KEY: credential used to authenticate traces in LangSmith
# OPENAI_API_KEY: credential used to authenticate requests to the OpenAI API
# (used for embeddings and/or chat completions in the RAG pipeline)

# Part 2: Query Translations
Query transformation refers to techniques that rephrase or adjust a question so it works better for retrieval.

## Part 2.1: Multi Query

### Index

In [ ]:
### Indexing

import bs4
from langchain_community.document_loaders import WebBaseLoader

# Download a web article
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Coffee",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(id=["firstHeading", "mw-content-text"])
    ),
)
blog_docs = loader.load()

# Split it in chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
  chunk_size=1000, # Each chunk 1000 characters
  chunk_overlap=200 # Each chunk repeat the 200 last characters of the previous one
)

splits = text_splitter.split_documents(blog_docs)

# Create a vector database from the chunks
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
  documents=splits, # Use the digested chunks as documents
  embedding=OpenAIEmbeddings() # Use OpenAI embeddings to vectorize the documents
)

retriever = vectorstore.as_retriever()

### Prompting

The prompt asks the LLM to rewrite the user's question in five different ways, one per line, to improve retrieval.
The chain fills the template, calls the model, extracts the text and splits it into a list of five queries.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.prompts import ChatPromptTemplate

# First we have to generate the different perspectives of the question,
# so we can retrieve more relevant passages from the vector database
template = """You are a helpful assistant that improves document search. Rewrite the question below in {number}
different ways, each one approaching the topic from a distinct angle or using different wording.
The goal is to retrieve as many relevant passages as possible from a vector database,
even when the original phrasing does not match the text closely. Return only the {number} rewritten questions,
one per line, with no numbering and no extra commentary. Question: {question}"""

# Transform the template into a prompt that can be used with a language model
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Generate the different perspectives of the question

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# The chain is built with LCEL: each "|" pipes the output of one step into the next
generate_queries = (
    prompt_perspectives # Fills {question} in the template and builds the chat messages for the model
    | ChatOpenAI(model_name="gpt-5-nano", temperature=0) # Calls the LLM; temperature=0 makes the output 0% random. Returns an AIMessage
    | StrOutputParser() # Extracts the plain text (str) from the AIMessage
    | (lambda x: x.split("\n")) # Splits the text into a list of strings, one per line (relies on the prompt asking for one question per line)
)

In this example, as it can be seen at LangSmith console, the 5 questions generated by OpenAI LLM are the following:
- What environmental impacts and social issues are associated with coffee production?
- What ecological effects and community-level challenges accompany coffee farming?
- Which environmental risks and labor-related or social consequences are linked to the coffee industry?
- What impacts on ecosystems and local populations arise from coffee cultivation and processing?
- What sustainability concerns—environmental and social—are connected to the coffee supply chain?

Now, having the questions, the next step is asking them to the LLM and join the answers

In [ ]:
from langchain_core.load import dumps, loads

# Receives a list of lists, where each list contains the questions, the documents retrieved.
def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Delete duplicates by converting to a set and back to a list
    unique_docs = list(set(flattened_docs))
    # Return in Document format again
    return [loads(doc) for doc in unique_docs]

# Retrieve
# Users possible question
question = "What are the environmental and social problems linked to coffee production?"

# Retrieval chain with Multi Query:
# 1) generate_queries: the LLM rewrites the question in 5 different ways
# 2) retriever.map(): searches Chroma with EACH rewrite (list of lists of chunks)
# 3) get_unique_union: flattens the result and removes duplicate chunks
retrieval_chain = generate_queries | retriever.map() | get_unique_union

# Executes the retrieval chain with the user question, and returns the retrieved documents
docs = retrieval_chain.invoke({"question":question, "number":5})
len(docs)

The last part is to take retrieved infromation by the model to each one of the questions

In [ ]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Call the model
llm = ChatOpenAI(
  model_name="gpt-5-nano", # NOV 2026 the more affordable OpenAI model is GPT 5 Nano
  temperature=0 # Want the model to be 0 creative
)


final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question, "number":5})